# 11 · ConceptNet 100k → Graphex W/S/I + residual → exact decode → Haken

**Часть 2 из 2.** Запускайте после ноутбука 10 с теми же параметрами и аккаунтом Drive. Строит ограниченный двухпортовый структурный словарь внутри GloVe–Wishart семейств, типизированную эмпирическую блоковую модель W, профили внешних коннекторов и точный архив W/S/I/R; сравнивает **полные байты** с независимым ZIP-baseline, выполняет точное декодирование и проверяет медленные спектральные моды.

Это конечная эмпирическая W/S/I-модель, **не** доказательство предельного графекса. Вариант без мотивов тоже проверяется: если словарь увеличивает архив, выбирается меньший вариант. Никакая экономия и сохранность динамики заранее не предполагаются.

In [ ]:
# Все пути — прежний каталог SemanticMap/colab/wishart на Google Drive.
BRANCH = "feature/glove-wishart-graphex-cn100k-lossless-l4"
DRIVE_ROOT = "/content/drive/MyDrive/SemanticMap/colab/wishart"
SOURCE_RUN = "wishart-dictionary-100k-gpu-gpu"  # Проверенный предыдущий run с N=100000.
RUN_NAME = "wishart-glove-graphex-cn100k-01"
GLOVE_REL = "data/glove.6B.100d.txt"  # Автозагрузка официального GloVe, если файла нет.
SCRATCH_ROOT = "/content/semmap-glove-graphex-100k"
EXPECTED_NODES = 100000
K_NEIGHBORS = 12
QUERY_BATCH = 256
MAX_FIGURES = 4096
MIN_SUPPORT = 5


In [ ]:
import os, sys, shutil, platform
from pathlib import Path
import torch
print("Python:", platform.python_version(), "CPUs:", os.cpu_count(),
      "free scratch GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
if not torch.cuda.is_available():
    raise RuntimeError("Для 100k GPU-kNN выберите Runtime → Change runtime type → L4 GPU")
DEVICE = torch.cuda.get_device_name(0)
print("GPU:", DEVICE, "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if "L4" not in DEVICE.upper(): print("WARNING: конфигурация проверялась для L4, GPU отличается")
if shutil.disk_usage("/content").free < 5 * 2**30:
    raise RuntimeError("Нужно >=5 GiB локального scratch")


In [ ]:
import os, sys, shutil, subprocess, json, hashlib
from pathlib import Path
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"  # Не переподписывать 12 CPU для BLAS плюс Python workers.
ROOT = Path(SCRATCH_ROOT); ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/SemanticMap/semgraphex.git", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
PROBE = subprocess.run([sys.executable, "-c", "import numpy, scipy, torch; from scipy import sparse; print(numpy.__version__, scipy.__version__, torch.cuda.is_available())"], capture_output=True, text=True)
if PROBE.returncode: raise RuntimeError(PROBE.stderr[-2500:])
print("Repo commit:", COMMIT, "dependency probe:", PROBE.stdout.strip())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path(DRIVE_ROOT)
SOURCE = DRIVE / "runs" / SOURCE_RUN / "level_000"
REMOTE = DRIVE / "runs" / RUN_NAME
LOCAL = ROOT / "runs" / RUN_NAME
LOCAL_INPUT = ROOT / "inputs" / "level_000"
GLOVE_REMOTE = DRIVE / GLOVE_REL
for required in (SOURCE / "adjacency.npz", SOURCE / "membership.json", SOURCE / "relations" / "index.json"):
    if not required.is_file(): raise FileNotFoundError(required)
if not (DRIVE / "runs" / SOURCE_RUN / "COMPLETED").is_file():
    raise RuntimeError("Исходный Wishart-run должен быть COMPLETED")
if not LOCAL_INPUT.exists(): shutil.copytree(SOURCE, LOCAL_INPUT)
from scipy import sparse
shape = sparse.load_npz(LOCAL_INPUT / "adjacency.npz").shape
if shape != (EXPECTED_NODES, EXPECTED_NODES): raise RuntimeError(f"Неверная выборка: {shape}")
REMOTE.mkdir(parents=True, exist_ok=True)
from semmap_haken.wishart_colab import stage_from_drive, sync_tree
if (REMOTE / "input.json").is_file():
    stage_from_drive(REMOTE, LOCAL, compare_mode="sha256")
else:
    LOCAL.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE, "shape:", shape, "output:", REMOTE)


In [ ]:
# GloVe нужен только для проверки SHA-256 идентичности входа с ноутбуком 10.
GLOVE = ROOT / "inputs" / "glove.6B.100d.txt"
if not GLOVE.is_file() or GLOVE.stat().st_size != GLOVE_REMOTE.stat().st_size:
    shutil.copy2(GLOVE_REMOTE, GLOVE)
if not (LOCAL / "checkpoints" / "wishart.json").is_file():
    raise RuntimeError("Сначала запустите ноутбук 10: нет checkpoint Wishart")
previous_revision = json.loads((LOCAL / "input.json").read_text(encoding="utf8"))["code_revision"]
if previous_revision != COMMIT:
    raise RuntimeError(f"Коммит изменился ({previous_revision} != {COMMIT}); установите тот же Git ref")
print("Input commit verified:", COMMIT)

In [ ]:
ENV = dict(os.environ, PYTHONPATH=str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
BASE = [sys.executable, "-m", "semmap_haken.glove_graphex_100k",
        "--source-level", str(LOCAL_INPUT), "--glove", str(GLOVE),
        "--run", str(LOCAL), "--checkpoint-drive", str(REMOTE),
        "--expected-nodes", str(EXPECTED_NODES), "--k", str(K_NEIGHBORS),
        "--batch", str(QUERY_BATCH), "--max-figures", str(MAX_FIGURES),
        "--min-support", str(MIN_SUPPORT)]
def run_stage(stage):
    print("====", stage, "====", flush=True)
    subprocess.run(BASE + ["--stage", stage], cwd=REPO, env=ENV, check=True)
    sync_tree(LOCAL, REMOTE, compare_mode="size_mtime", final=False)
    print("Checkpoint saved:", REMOTE / "checkpoints" / (stage + ".json"), flush=True)


In [ ]:
for stage in ("codec", "haken", "verify"):
    run_stage(stage)
# Публикуем COMPLETED только ПОСЛЕ синхронизации всех результатов.
sync_tree(LOCAL, REMOTE, compare_mode="sha256", final=True)
assert (REMOTE / "COMPLETED").is_file()
print("COMPLETE:", REMOTE)

In [ ]:
report = json.loads((LOCAL / "codec_report.json").read_text(encoding="utf8"))
haken = json.loads((LOCAL / "dynamics" / "haken.json").read_text(encoding="utf8"))
print(json.dumps({"archive_bytes": report["selected_archive_bytes"],
                  "baseline_bytes": report["baseline_bytes"],
                  "net_saved_bytes": report["net_saved_bytes"],
                  "archive_ratio": report["ratio"],
                  "roundtrip": json.loads((LOCAL / "verification.json").read_text())["lossless"],
                  "quotient_nodes": haken["quotient_nodes"],
                  "stationary_mass_TV": haken["stationary_mass_total_variation"],
                  "principal_angles_degrees": haken["nontrivial_principal_angles_degrees"]}, indent=2))